In [10]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer

In [36]:
import spacy
import re
from nltk.corpus import wordnet
import nltk
nltk.download('wordnet')
nltk.download('omw-1.4')


[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\RobotComp.ru\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\RobotComp.ru\AppData\Roaming\nltk_data...


In [40]:
nlp_ru = spacy.load("ru_core_news_sm")
nlp_en = spacy.load("en_core_web_sm")

In [8]:
df = pd.read_csv('task10.csv')


In [45]:
stop_words = set(nltk.corpus.stopwords.words('russian')) | set(nltk.corpus.stopwords.words('english'))

def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'[^а-яА-Яa-zA-Z0-9\s]', ' ', text)

    tokens_ru = []
    tokens_en = []

    doc_ru = nlp_ru(text)
    doc_en = nlp_en(text)

    for token in doc_ru:
        if token.is_alpha and token.text not in stop_words:
            tokens_ru.append(token.lemma_)

    for token in doc_en:
        if token.is_alpha and token.text not in stop_words:
            tokens_en.append(token.lemma_)

    expanded_en = tokens_en.copy()

    expanded_ru = tokens_ru.copy()

    all_tokens = list(set(expanded_ru + expanded_en))
    return all_tokens

In [46]:
texts = df["question"] 
texts = texts.apply(preprocess_text)
texts = texts.apply(lambda x: ' '.join(x))
vectorizer = CountVectorizer()

X = vectorizer.fit_transform(texts)
questions = df["question"]
answers = df["answer"]
print("Размерность матрицы Bag of Words:", X.shape)


Размерность матрицы Bag of Words: (38, 139)


In [48]:
bow_df = pd.DataFrame(X.toarray(), columns=vectorizer.get_feature_names_out())
print(bow_df.head())
lemma_counts = bow_df.sum(axis=0)

lemma_counts_df = pd.DataFrame(lemma_counts, columns=['count']).sort_values(by='count', ascending=False)

print(lemma_counts_df.head(20)) 

   args  bytes  comprehension  copy  deepcopy  except  gil  kwargs  lambda  \
0     0      0              0     1         1       0    0       0       0   
1     0      0              0     0         0       0    0       0       0   
2     1      0              0     0         0       0    0       1       0   
3     0      0              0     0         0       0    1       0       0   
4     0      0              0     0         0       0    0       0       1   

   list  ...  умолчание  умолчанию  файл  файлы  функции  функцию  функция  \
0     0  ...          0          0     0      0        0        0        0   
1     0  ...          0          0     0      0        0        0        0   
2     0  ...          0          0     0      0        0        0        0   
3     0  ...          0          0     0      0        0        0        0   
4     0  ...          0          0     0      0        1        0        1   

   цикл  является  являться  
0     0         0         0  
1 

In [61]:
def jaccard_similarity_tokens(text1, text2):
    tokens1 = set(text1.lower().split())
    tokens2 = set(text2.lower().split())

    A = len(tokens1)
    B = len(tokens2)
    C = len(tokens1.intersection(tokens2))

    return C / (A + B - C) if (A + B - C) != 0 else 0

In [15]:
import numpy as np
from numpy.linalg import norm

def cosine_distance(v1, v2):
    v1 = v1.toarray()[0]
    v2 = v2.toarray()[0]

    dot = np.dot(v1, v2)
    normA = norm(v1)
    normB = norm(v2)

    if normA == 0 or normB == 0:
        return 1  # максимальная дистанция

    cosine_similarity = dot / (normA * normB)
    cosine_distance = 1 - cosine_similarity

    return cosine_distance

In [16]:
from scipy.stats import pearsonr
def correlation_similarity(v1, v2):
    v1 = v1.toarray()[0]
    v2 = v2.toarray()[0]
    if np.std(v1) == 0 or np.std(v2) == 0:
        return 0
    return pearsonr(v1, v2)[0]

In [103]:

def find_best_match(user_question):
    # Вектор для пользовательского вопроса
    user_question = " ".join(preprocess_text(user_question))
    user_vec = vectorizer.transform([user_question])

    best_scores = {
        "Косинус": {"score": 2, "index": -1},
        "Жаккар": {"score": -1, "index": -1},
        "Корреляция": {"score": -1, "index": -1},
        "Среднее": {"score": -1, "index": -1}
    }
    for i in range(len(questions)):
        q_vec = X[i]

        cos = cosine_distance(user_vec, q_vec)

        jac = jaccard_similarity_tokens(user_question, questions[i])

        corr = correlation_similarity(user_vec, q_vec)

        final_score = (cos + jac + corr) / 3

        if cos < best_scores["Косинус"]["score"]:
            best_scores["Косинус"]["score"] = cos
            best_scores["Косинус"]["index"] = i

        if jac > best_scores["Жаккар"]["score"]:
            best_scores["Жаккар"]["score"] = jac
            best_scores["Жаккар"]["index"] = i

        if corr > best_scores["Корреляция"]["score"]:
            best_scores["Корреляция"]["score"] = corr
            best_scores["Корреляция"]["index"] = i

        # Среднее
        avg_score = (-cos + jac + corr) / 3
        if avg_score > best_scores["Среднее"]["score"]:
            best_scores["Среднее"]["score"] = avg_score
            best_scores["Среднее"]["index"] = i

    results = {}
    for method, info in best_scores.items():
        idx = info["index"]
        results[method] = {
            "best_question": questions[idx],
            "best_answer": answers[idx],
            "score": info["score"]
        }

    return results


In [50]:
query = "Почем в Python  использовать виртуальные ?"
result = find_best_match(query)
print(result)

{'best_question': 'Почему в Python важно использовать виртуальные окружения?', 'best_answer': 'Чтобы изолировать зависимости проекта и избежать конфликтов версий библиотек между разными проектами.', 'score': np.float64(0.47718837437232237), 'res_score': (0.0, 1.0, -0.02868955263161021)}


In [78]:
def testing(test_queries):
    for query in test_queries:
        result = find_best_match(query)
        print(f"ВОПРОС: {query}")
        print("-" * 50)
        for method, res in result.items():
            print(f"Метод: {method}")
            print(f"Лучший вопрос: {res['best_question']}")
            print(f"Ответ: {res['best_answer']}")
            print(f"Оценка сходства: {res['score']}")
            print("-" * 50)

In [82]:
import pandas as pd

def testing_table(test_queries):
    rows = []

    for query in test_queries:
        result = find_best_match(query) 
        row = {"Query": query}

        for method, res in result.items():
            row[f"{method}_best_question"] = res['best_question']
            row[f"{method}_answer"] = res['best_answer']
            row[f"{method}_score"] = res['score']

        rows.append(row)

    df_results = pd.DataFrame(rows)
    return df_results



In [85]:
import matplotlib.pyplot as plt
def plot_similarity_scores(df_results):
    methods = ['Косинус', 'Жаккар', 'Корреляция', 'Среднее']

    queries = df_results['Query']

    # Для каждого метода проверяем совпадение с query
    accuracy_matrix = {m: [] for m in methods}
    for i, query in enumerate(queries):
        for m in methods:
            best_question = df_results.loc[i, f"{m}_best_question"]
            # True если метод выбрал правильный вопрос
            accuracy_matrix[m].append(int(best_question == query))

    # График
    x = range(len(queries))
    width = 0.2

    plt.figure(figsize=(12, 6))
    for i, m in enumerate(methods):
        plt.bar([xi + i*width for xi in x], accuracy_matrix[m], width=width, label=m.capitalize())

    plt.xticks([xi + width*1.5 for xi in x], queries, rotation=30, ha='right')
    plt.ylabel("Верно предсказано (1 = Да, 0 = Нет)")
    plt.title("Сравнение методов по точности предсказания")
    plt.ylim(0, 1.2)
    plt.legend()
    plt.tight_layout()
    plt.show()

In [87]:
from sklearn.metrics import accuracy_score, precision_score, recall_score

def evaluate_methods(df_results):
    methods = ['Косинус', 'Жаккар', 'Корреляция', 'Среднее']
    queries = df_results['Query']

    metrics = {}

    for m in methods:
        y_true = [1]*len(queries)  # правильный ответ = 1 для всех
        y_pred = [1 if df_results.loc[i, f"{m}_best_question"] == queries[i] else 0 for i in range(len(queries))]

        acc = accuracy_score(y_true, y_pred)
        prec = precision_score(y_true, y_pred)
        rec = recall_score(y_true, y_pred)

        metrics[m] = {"Accuracy": acc, "Precision": prec, "Recall": rec}

    return metrics

# Использование



In [104]:

test_queries = [
    "В чем различия между deepcopy и copy?",
    "Зачем в питоне нужны словари?",
    "Как работает распаковка значений?",
    "Зачем в Python  нужнен GIL?",
    "Зачем нужны lambda функции?"
]

testing(test_queries)





ВОПРОС: В чем различия между deepcopy и copy?
--------------------------------------------------
Метод: Косинус
Лучший вопрос: Почему важно понимать различия между deepcopy и copy?
Ответ: copy создаёт поверхностную копию, а deepcopy дублирует вложенные объекты; неправильный выбор приводит к неожиданным изменениям данных.
Оценка сходства: 0.20259099165917982
--------------------------------------------------
Метод: Жаккар
Лучший вопрос: Почему важно понимать различия между deepcopy и copy?
Ответ: copy создаёт поверхностную копию, а deepcopy дублирует вложенные объекты; неправильный выбор приводит к неожиданным изменениям данных.
Оценка сходства: 0.2
--------------------------------------------------
Метод: Корреляция
Лучший вопрос: Почему важно понимать различия между deepcopy и copy?
Ответ: copy создаёт поверхностную копию, а deepcopy дублирует вложенные объекты; неправильный выбор приводит к неожиданным изменениям данных.
Оценка сходства: 0.7908075966688584
---------------------------

In [105]:
test_queries = [
    "В чем отличие deepcopy от copy в Python?",
    "Почему в Python важно использовать словари?",
    "Как в Python работает распаковка аргументов?",
    "Почему в Python существует глобальная блокировка интерпретатора?",
    "Для чего используют функции lambda в Python?"
]
testing(test_queries)

ВОПРОС: В чем отличие deepcopy от copy в Python?
--------------------------------------------------
Метод: Косинус
Лучший вопрос: Почему важно понимать различия между deepcopy и copy?
Ответ: copy создаёт поверхностную копию, а deepcopy дублирует вложенные объекты; неправильный выбор приводит к неожиданным изменениям данных.
Оценка сходства: 0.47710905673317017
--------------------------------------------------
Метод: Жаккар
Лучший вопрос: Зачем в Python нужны словари?
Ответ: Чтобы быстро хранить и находить данные по уникальному ключу с амортизированной сложностью O(1).
Оценка сходства: 0.125
--------------------------------------------------
Метод: Корреляция
Лучший вопрос: Почему важно понимать различия между deepcopy и copy?
Ответ: copy создаёт поверхностную копию, а deepcopy дублирует вложенные объекты; неправильный выбор приводит к неожиданным изменениям данных.
Оценка сходства: 0.5084805449759278
--------------------------------------------------
Метод: Среднее
Лучший вопрос: Поче

In [94]:
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd

texts = df["question"]
texts = texts.apply(preprocess_text)
texts = texts.apply(lambda x: " ".join(x))
vectorizer = TfidfVectorizer(lowercase=True, stop_words='english')  

X = vectorizer.fit_transform(texts)
questions = df["question"]
answers = df["answer"]
print("Размерность TF-IDF матрицы:", X.shape)

tfidf_df = pd.DataFrame(X.toarray(), columns=vectorizer.get_feature_names_out())
print(tfidf_df.head())

Размерность TF-IDF матрицы: (38, 138)
       args  bytes  comprehension      copy  deepcopy       gil    kwargs  \
0  0.000000    0.0            0.0  0.398705  0.398705  0.000000  0.000000   
1  0.000000    0.0            0.0  0.000000  0.000000  0.000000  0.000000   
2  0.509481    0.0            0.0  0.000000  0.000000  0.000000  0.509481   
3  0.000000    0.0            0.0  0.000000  0.000000  0.388464  0.000000   
4  0.000000    0.0            0.0  0.000000  0.000000  0.000000  0.000000   

     lambda  list  log  logging  pep  pip  print    python  return  self  str  \
0  0.000000   0.0  0.0      0.0  0.0  0.0    0.0  0.000000     0.0   0.0  0.0   
1  0.000000   0.0  0.0      0.0  0.0  0.0    0.0  0.292226     0.0   0.0  0.0   
2  0.000000   0.0  0.0      0.0  0.0  0.0    0.0  0.000000     0.0   0.0  0.0   
3  0.000000   0.0  0.0      0.0  0.0  0.0    0.0  0.221672     0.0   0.0  0.0   
4  0.424564   0.0  0.0      0.0  0.0  0.0    0.0  0.000000     0.0   0.0  0.0   

   try  venv

In [106]:
test_queries = [
    "В чем отличие deepcopy от copy в Python?",
    "Почему в Python важно использовать словари?",
    "Как в Python работает распаковка аргументов?",
    "Почему в Python существует глобальная блокировка интерпретатора (GIL)?",
    "Для чего используют функции lambda в Python?"
]
testing(test_queries)

ВОПРОС: В чем отличие deepcopy от copy в Python?
--------------------------------------------------
Метод: Косинус
Лучший вопрос: Почему важно понимать различия между deepcopy и copy?
Ответ: copy создаёт поверхностную копию, а deepcopy дублирует вложенные объекты; неправильный выбор приводит к неожиданным изменениям данных.
Оценка сходства: 0.47710905673317017
--------------------------------------------------
Метод: Жаккар
Лучший вопрос: Зачем в Python нужны словари?
Ответ: Чтобы быстро хранить и находить данные по уникальному ключу с амортизированной сложностью O(1).
Оценка сходства: 0.125
--------------------------------------------------
Метод: Корреляция
Лучший вопрос: Почему важно понимать различия между deepcopy и copy?
Ответ: copy создаёт поверхностную копию, а deepcopy дублирует вложенные объекты; неправильный выбор приводит к неожиданным изменениям данных.
Оценка сходства: 0.5084805449759278
--------------------------------------------------
Метод: Среднее
Лучший вопрос: Поче

In [107]:
print("Режим диалога с базой знаний Python")
print("Введите 'выход', чтобы завершить.")

while True:
    query = input("\nВаш вопрос: \n")
    if query.lower() in ["q"]:
        print("Диалог завершен.")
        break

    results = find_best_match(query)

    for method, res in results.items():
        print(f"\nМетод: {method}")
        print(f"Лучший вопрос: {res['best_question']}")
        print(f"Ответ: {res['best_answer']}")
        print(f"Оценка сходства: {res['score']:.3f}")

=== Режим диалога с базой знаний Python ===
Введите 'выход', чтобы завершить.

Метод: Косинус
Лучший вопрос: Зачем в Python нужны словари?
Ответ: Чтобы быстро хранить и находить данные по уникальному ключу с амортизированной сложностью O(1).
Оценка сходства: 0.619

Метод: Жаккар
Лучший вопрос: Как создать объект класса?
Ответ: Вызвать класс как функцию: obj = MyClass().
Оценка сходства: 0.200

Метод: Корреляция
Лучший вопрос: Зачем в Python нужны словари?
Ответ: Чтобы быстро хранить и находить данные по уникальному ключу с амортизированной сложностью O(1).
Оценка сходства: 0.368

Метод: Среднее
Лучший вопрос: Как создать объект класса?
Ответ: Вызвать класс как функцию: obj = MyClass().
Оценка сходства: -0.048
Диалог завершен.
